# iPerturb — reproducibility notebook

Runs the full **iPerturb** pipeline (build context-specific GRN → fit Hill-kinetics GRNN → evaluate) for **K562** and **RPE1**, from a clean clone to a downloadable results zip.

**Before you start:** set a **GPU** runtime (`Runtime → Change runtime type → GPU`).
Expected runtime ~30–60 min (database downloads + training for two cell lines).

You must supply the **GeneHancer v5.26** files yourself (license-gated, not redistributable) — see step 4.

In [ ]:
# 1. Check GPU and clone the repository
!nvidia-smi -L || echo 'No GPU detected — enable one via Runtime > Change runtime type > GPU'
![ -d /content/iPerturb/.git ] || git clone https://github.com/kagtgi/iPerturb.git /content/iPerturb
%cd /content/iPerturb

In [ ]:
# 2. Install dependencies (Colab already has torch/numpy/pandas/etc.; the rest install here)
!pip install -q -r requirements.txt

In [ ]:
# 3. Download the Replogle Perturb-seq expression data into /content
import os, urllib.request
FILES = {'K562.h5ad': '35773219', 'RPE1.h5ad': '35775606'}
for fn, fid in FILES.items():
    dst = f'/content/{fn}'
    if not (os.path.exists(dst) and os.path.getsize(dst) > 1e6):
        print('Downloading', fn, '...')
        urllib.request.urlretrieve(f'https://ndownloader.figshare.com/files/{fid}', dst)
    print(fn, f'{os.path.getsize(dst)/1e6:.0f} MB')

## 4. GeneHancer v5.26 (required, license-gated)

GeneHancer is **not redistributable**, so it cannot ship with the repo. Supply the three files (`GeneHancer_v5.26.gff`, `GeneHancer_TFBSs_v5.26.txt`, `GeneHancer_Tissues_v5.26.txt`, ~605 MB total) from your own licensed copy via Google Drive:

* **Option A** — put the 3 files in a Drive folder and set `GENEHANCER_DIR`.
* **Option B** — put a single `.zip` of them in Drive and set `GENEHANCER_ZIP` (faster to upload once).

The cell mounts Drive, copies/extracts them to `/content/`, and checks their sizes.

In [ ]:
# 4. Make the GeneHancer v5.26 files available under /content (license-gated; user-provided)
GENEHANCER_DIR = '/content/drive/MyDrive/GeneHancer'   # folder containing the 3 files
GENEHANCER_ZIP = ''                                    # OR a single .zip on Drive, e.g. '/content/drive/MyDrive/GeneHancer.zip'

import os, shutil, zipfile, glob
NEEDED = ['GeneHancer_v5.26.gff', 'GeneHancer_TFBSs_v5.26.txt', 'GeneHancer_Tissues_v5.26.txt']
def _ok(f): return os.path.exists(f'/content/{f}') and os.path.getsize(f'/content/{f}') > 1e6

if not all(_ok(f) for f in NEEDED):
    if GENEHANCER_DIR or GENEHANCER_ZIP:
        try:
            from google.colab import drive; drive.mount('/content/drive')
        except Exception as e:
            print('Drive mount skipped:', e)
    if GENEHANCER_ZIP and os.path.exists(GENEHANCER_ZIP):
        print('Extracting', GENEHANCER_ZIP, '...')
        with zipfile.ZipFile(GENEHANCER_ZIP) as z: z.extractall('/content')
        for f in NEEDED:                       # flatten if the zip had a subfolder
            if not _ok(f):
                hits = glob.glob(f'/content/**/{f}', recursive=True)
                if hits: shutil.copy(hits[0], f'/content/{f}')
    elif GENEHANCER_DIR:
        for f in NEEDED:
            src = os.path.join(GENEHANCER_DIR, f)
            if os.path.exists(src) and not _ok(f):
                print('copying', f, f'({os.path.getsize(src)/1e6:.0f} MB) ...'); shutil.copy(src, f'/content/{f}')

missing = [f for f in NEEDED if not _ok(f)]
assert not missing, ('Missing/incomplete GeneHancer files: ' + str(missing) +
                     '
Put the 3 files in ' + GENEHANCER_DIR + ' (or set GENEHANCER_ZIP), or upload them to /content/.')
for f in NEEDED:
    print(f, f"{os.path.getsize(f'/content/{f}')/1e6:.0f} MB OK")

## 5. Run the full pipeline (K562, then RPE1)
This builds each GRN, trains the GRNN, evaluates on held-out perturbations, and writes figures + metrics under `/content/`. Takes a while on GPU.

In [ ]:
# 5. Run end-to-end
%run /content/iPerturb/iperturb.py

## 6. Collect and download results

In [ ]:
# 6. Zip the outputs (figures + metrics + GRN tables) and download
import os, glob, shutil
os.makedirs('/content/results', exist_ok=True)
if os.path.isdir('/content/grn_plots'):
    shutil.copytree('/content/grn_plots', '/content/results/figures', dirs_exist_ok=True)
PATTERNS = ['/content/*_metrics_subsample_*runs.tsv', '/content/*_grn_logged.tsv',
            '/content/grn_edges.tsv', '/content/grn_full.graphml', '/content/grn_gene_lists.txt']
for pat in PATTERNS:
    for p in glob.glob(pat):
        shutil.copy(p, '/content/results/')
shutil.make_archive('/content/iperturb_results', 'zip', '/content/results')
print('Wrote /content/iperturb_results.zip')
try:
    from google.colab import files
    files.download('/content/iperturb_results.zip')
except Exception as e:
    print('Download manually from the Files panel:', e)